In [1]:
import numpy as np
import sys
sys.path.append("build/Release")
import matplotlib.pyplot as plt
import FXLIB as fx
from time import perf_counter

In [2]:
def RGB_to_np(rgb):
    return np.array(rgb.to_array())

def generate_image(func, t, size=(400, 400), scale = 0.1):
    # output = np.zeros([size[0], size[1], 3])
    
    x, y = np.meshgrid(np.linspace(-scale, scale, size[0]), np.linspace(-scale, scale, size[1]))
    x = x.flatten()
    y = y.flatten()
    begin = perf_counter()
    raw_data = fx.batch_process_array(func, x, y, t)
    end = perf_counter()
    print(f"Time to process effect {end-begin}")

    begin = perf_counter()
    data = np.array(raw_data).reshape((size[0], size[1], 3))
    end = perf_counter()
    print(f"Time to cast to np array {end-begin}")

    # Reshape into the desired output shape (height, width, 3)
    return data

def imshow(func, t, size=(400, 400), scale=0.1):
    plt.imshow(generate_image(func, t, size, scale))

In [4]:
"""
Demonstrates very basic use of ImageItem to display image data inside a ViewBox.
"""

from time import perf_counter

import numpy as np

import pyqtgraph as pg
from pyqtgraph.Qt import QtCore

app = pg.mkQApp("Animation")

## Create window with GraphicsView widget
win = pg.GraphicsLayoutWidget()
win.show()  ## show widget alone in its own window
win.setWindowTitle('Animation')
view = win.addViewBox()

view.setAspectLocked(True)

## Create image item
img = pg.ImageItem(border='w')
view.addItem(img)

## Set initial view bounds
view.setRange(QtCore.QRectF(0, 0, 600, 600))

## Create random image
data = np.random.normal(size=(15, 600, 600), loc=1024, scale=64).astype(np.uint16)
t = 0

updateTime = perf_counter()
elapsed = 0

timer = QtCore.QTimer()
timer.setSingleShot(True)
# not using QTimer.singleShot() because of persistence on PyQt. see PR #1605

func = fx.cycle(fx.rainbow, 0.1)

def updateData():
    global img, data, t, updateTime, elapsed

    t += 0.1
    data = generate_image(func, t)
    ## Display the data
    img.setImage(data)

    timer.start(1)
    now = perf_counter()
    elapsed_now = now - updateTime
    updateTime = now
    elapsed = elapsed * 0.9 + elapsed_now * 0.1

    # print(f"{1 / elapsed:.1f} fps")
    
timer.timeout.connect(updateData)
updateData()

if __name__ == '__main__':
    pg.exec()


In [4]:
func = fx.solid(fx.red)
from time import perf_counter
size = (1000, 1000)
scale = 1.0
t = 0
count = (size[0]*size[1])

x = y = np.linspace(-1, 1, count)
start = perf_counter()
print(x.data)
fx.generate_image(t, x, y)
end = perf_counter()
print(f"Time: {end-start}")

Time: 0.1294796000001952


In [9]:
dir(fx.solid)

['__call__',
 '__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '_pybind11_conduit_v1_',
 'color']